In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os
import numpy as np
import librosa
from tqdm import tqdm

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/IEMOCAPdataset"

print("Top level folders:")
print(os.listdir(dataset_path))

In [ ]:
!tar -xvzf "/content/drive/MyDrive/IEMOCAPdataset/IEMOCAP_full_release_withoutVideos.tar.gz" -C /content/

In [ ]:
dataset_path = "/content/IEMOCAP_full_release"

In [ ]:
import os
import glob
from collections import defaultdict

dataset_path = "/content/IEMOCAP_full_release"

# 1. COUNT AUDIO FILES (.wav)
wav_files = glob.glob(os.path.join(dataset_path, "**", "*.wav"), recursive=True)
print(f" Total Audio Files (.wav): {len(wav_files)}")

# 2. COUNT TEXT FILES (.txt)
txt_files = glob.glob(os.path.join(dataset_path, "**", "*.txt"), recursive=True)
print(f" Total Text Files (.txt): {len(txt_files)}")

# 3. COUNT CONVERSATIONS
# Conversations are stored in:
# Session*/dialog/transcriptions/*.txt

conversation_files = glob.glob(
    os.path.join(dataset_path, "Session*", "dialog", "transcriptions", "*.txt")
)

print(f"💬 Total Conversations: {len(conversation_files)}")

# 4. COUNT SESSIONS
sessions = glob.glob(os.path.join(dataset_path, "Session*"))
print(f"Total Sessions: {len(sessions)}")

# 5. CONVERSATIONS PER SESSION
session_conv_count = {}

for session in sessions:
    session_name = os.path.basename(session)
    convs = glob.glob(
        os.path.join(session, "dialog", "transcriptions", "*.txt")
    )
    session_conv_count[session_name] = len(convs)

print("\nConversations per Session:")
for k, v in sorted(session_conv_count.items()):
    print(f"{k}: {v}")

# 6. EMOTION LABEL FILES (.txt)
# Emotion annotations are in:
# Session*/dialog/EmoEvaluation/*.txt

emo_files = glob.glob(
    os.path.join(dataset_path, "Session*", "dialog", "EmoEvaluation", "*.txt")
)

print(f"\nEmotion Annotation Files: {len(emo_files)}")

# 7. AUDIO FILES PER SESSION
session_audio_count = defaultdict(int)

for wav in wav_files:
    for session in sessions:
        if session in wav:
            session_name = os.path.basename(session)
            session_audio_count[session_name] += 1

print("\n🎧 Audio files per Session:")
for k, v in sorted(session_audio_count.items()):
    print(f"{k}: {v}")

# 8. SAMPLE CONVERSATION FILE
print("\nSample Conversation Preview:\n")

if len(conversation_files) > 0:
    sample_file = conversation_files[0]
    print(f"File: {sample_file}\n")

    with open(sample_file, "r") as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i > 10:
                break
else:
    print("No conversation files found!")

# 9. TOTAL UTTERANCES (from transcriptions)
total_utterances = 0

for file in conversation_files:
    with open(file, "r") as f:
        total_utterances += len(f.readlines())

print(f"\nTotal Utterances: {total_utterances}")

# 10. SUMMARY
print("\n================ DATASET SUMMARY ================")
print(f"Sessions              : {len(sessions)}")
print(f"Conversations         : {len(conversation_files)}")
print(f"Audio files           : {len(wav_files)}")
print(f"Text files            : {len(txt_files)}")
print(f"Emotion label files   : {len(emo_files)}")
print(f"Total utterances      : {total_utterances}")
print("================================================")

**Mltimodal - MLP ( Audio features - Wav2Vec2 , Text features - ROBERTA)**

In [ ]:
# FINAL MULTIMODAL (WAV2VEC2 + ROBERTA + FUSION)

!pip install transformers librosa -q

import os, re, glob, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import Wav2Vec2Processor, Wav2Vec2Model
from transformers import RobertaTokenizer, RobertaModel

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATA
DATA_PATH = "/content/IEMOCAP_full_release"
valid_emotions = ["ang","hap","sad","neu","exc"]

wav_paths, texts, labels = [], [], []

for session in range(1,6):
    session_path = os.path.join(DATA_PATH, f"Session{session}")
    wav_files = glob.glob(os.path.join(session_path, "**/*.wav"), recursive=True)

    for wav_file in wav_files:
        file_id = os.path.basename(wav_file).replace(".wav","")
        parts = file_id.split("_")

        ann_name = "_".join(parts[:2]) + ".txt" if "impro" in file_id else "_".join(parts[:3]) + ".txt"
        ann_path = os.path.join(session_path,"dialog","EmoEvaluation",ann_name)
        trans_path = os.path.join(session_path,"dialog","transcriptions",ann_name)

        if not os.path.exists(ann_path) or not os.path.exists(trans_path):
            continue

        # emotion
        emotion = None
        with open(ann_path) as f:
            for line in f:
                if file_id in line:
                    emotion = line.strip().split()[4]
                    break

        if emotion not in valid_emotions:
            continue
        if emotion == "exc":
            emotion = "hap"

        # text
        text = ""
        with open(trans_path) as f:
            for line in f:
                if file_id in line:
                    text = " ".join(line.strip().split()[2:])
                    break

        if text == "":
            continue

        wav_paths.append(wav_file)
        texts.append(text)
        labels.append(emotion)

print("Samples:", len(wav_paths))

# ENCODING
le = LabelEncoder()
y = le.fit_transform(labels)

# SPLIT (IMPORTANT)
Xa_train_path, Xa_test_path, Xt_train, Xt_test, y_train, y_test = train_test_split(
    wav_paths, texts, y, test_size=0.2, stratify=y, random_state=42
)

# AUDIO: WAV2VEC2
wav_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
wav_model.eval()

def get_audio_embedding(path):
    signal, sr = librosa.load(path, sr=16000)
    inputs = wav_processor(signal, sampling_rate=16000, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = wav_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

# TEXT: ROBERTA
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
text_model = RobertaModel.from_pretrained("roberta-base").to(device)
text_model.eval()

def get_text_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
    return outputs.last_hidden_state[:,0,:].cpu().numpy().flatten()

# FEATURE EXTRACTION
Xa_train, Xt_train_vec = [], []
Xa_test, Xt_test_vec = [], []

print("Extracting TRAIN features...")
for a,t in tqdm(zip(Xa_train_path, Xt_train), total=len(Xa_train_path)):
    Xa_train.append(get_audio_embedding(a))
    Xt_train_vec.append(get_text_embedding(t))

print("Extracting TEST features...")
for a,t in tqdm(zip(Xa_test_path, Xt_test), total=len(Xa_test_path)):
    Xa_test.append(get_audio_embedding(a))
    Xt_test_vec.append(get_text_embedding(t))

Xa_train = np.array(Xa_train)
Xt_train_vec = np.array(Xt_train_vec)
Xa_test = np.array(Xa_test)
Xt_test_vec = np.array(Xt_test_vec)

# DATASET
train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(Xa_train, dtype=torch.float32),
    torch.tensor(Xt_train_vec, dtype=torch.float32),
    torch.tensor(y_train)
)

test_dataset = torch.utils.data.TensorDataset(
    torch.tensor(Xa_test, dtype=torch.float32),
    torch.tensor(Xt_test_vec, dtype=torch.float32),
    torch.tensor(y_test)
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16)

# CLASS WEIGHTS
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw = torch.tensor(cw, dtype=torch.float32).to(device)

# MODEL
class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_fc = nn.Sequential(nn.Linear(768,256), nn.ReLU())
        self.text_fc = nn.Sequential(nn.Linear(768,256), nn.ReLU())

        self.classifier = nn.Sequential(
            nn.Linear(512,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, len(le.classes_))
        )

    def forward(self, a, t):
        a = self.audio_fc(a)
        t = self.text_fc(t)
        x = torch.cat([a,t], dim=1)
        return self.classifier(x)

model = FusionNet().to(device)

criterion = nn.CrossEntropyLoss(weight=cw)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TRAINING
for epoch in range(8):
    model.train()
    total_loss = 0

    for xa, xt, yb in train_loader:
        xa, xt, yb = xa.to(device), xt.to(device), yb.to(device)

        outputs = model(xa, xt)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# EVALUATION
model.eval()
all_preds = []

with torch.no_grad():
    for xa, xt, _ in test_loader:
        xa, xt = xa.to(device), xt.to(device)
        preds = torch.argmax(model(xa, xt), dim=1)
        all_preds.extend(preds.cpu().numpy())

print("\nAccuracy:", accuracy_score(y_test, all_preds))
print(classification_report(y_test, all_preds))

pred_labels = le.inverse_transform(all_preds)

# EMOTION TREND ANALYSIS

records = []

for idx, (path, pred) in enumerate(zip(Xa_test_path, pred_labels)):
    filename = os.path.basename(path).replace(".wav","")
    parts = filename.split("_")

    conv_id = "_".join(parts[:2])
    match = re.search(r'(\d+)$', filename)
    utt_id = int(match.group(1)) if match else idx

    records.append({
        "conversation_id": conv_id,
        "utterance_id": utt_id,
        "emotion_label": pred
    })

df = pd.DataFrame(records).sort_values(by=["conversation_id","utterance_id"])

# VAD MAP
VAD_MAP = {
    "hap": (0.8,0.7,0.6),
    "sad": (-0.7,0.3,0.4),
    "ang": (-0.6,0.8,0.7),
    "neu": (0.0,0.5,0.5)
}

print("\nSample Conversation IDs:")
print(df["conversation_id"].unique()[:10])

cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val, label="Valence")
    plt.plot(aro, linestyle="--", label="Arousal")
    plt.plot(dom, linestyle=":", label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1] - val[0]
    variance = np.var(val)

    print("\nCONCLUSION:")

    if trend > 0.2:
        print("Positive Shift")
    elif trend < -0.2:
        print("Negative Shift")
    else:
        print("Stable Emotion")

    if variance > 0.5:
        print("High Fluctuation")
    else:
        print("Stable Behavior")

else:
    print("Conversation not found")

In [ ]:
cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val, label="Valence")
    plt.plot(aro, linestyle="--", label="Arousal")
    plt.plot(dom, linestyle=":", label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1] - val[0]
    variance = np.var(val)

    print("\nCONCLUSION:")

    if trend > 0.2:
        print("Positive Shift")
    elif trend < -0.2:
        print("Negative Shift")
    else:
        print("Stable Emotion")

    if variance > 0.5:
        print("High Fluctuation")
    else:
        print("Stable Behavior")

else:
    print("Conversation not found")

**Mltimodal - MLP ( Audio features - WavLM , Text features - DistilBERT)**

In [ ]:
# MULTIMODAL (WavLM + DistilBERT + Gated Fusion + BiLSTM)

!pip install transformers librosa -q

import os, re, glob, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import WavLMModel, AutoFeatureExtractor
from transformers import DistilBertTokenizer, DistilBertModel

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATA
DATA_PATH = "/content/IEMOCAP_full_release"
valid_emotions = ["ang","hap","sad","neu","exc"]

wav_paths, texts, labels = [], [], []

for session in range(1,6):
    session_path = os.path.join(DATA_PATH, f"Session{session}")
    wav_files = glob.glob(os.path.join(session_path, "**/*.wav"), recursive=True)

    for wav_file in wav_files:
        file_id = os.path.basename(wav_file).replace(".wav","")
        parts = file_id.split("_")

        ann_name = "_".join(parts[:2]) + ".txt" if "impro" in file_id else "_".join(parts[:3]) + ".txt"
        ann_path = os.path.join(session_path,"dialog","EmoEvaluation",ann_name)
        trans_path = os.path.join(session_path,"dialog","transcriptions",ann_name)

        if not os.path.exists(ann_path) or not os.path.exists(trans_path):
            continue

        emotion = None
        with open(ann_path) as f:
            for line in f:
                if file_id in line:
                    emotion = line.strip().split()[4]
                    break

        if emotion not in valid_emotions:
            continue
        if emotion == "exc":
            emotion = "hap"

        text = ""
        with open(trans_path) as f:
            for line in f:
                if file_id in line:
                    text = " ".join(line.strip().split()[2:])
                    break

        if text == "":
            continue

        wav_paths.append(wav_file)
        texts.append(text)
        labels.append(emotion)

print("Samples:", len(wav_paths))

# ENCODING
le = LabelEncoder()
y = le.fit_transform(labels)

Xa_train, Xa_test, Xt_train, Xt_test, y_train, y_test = train_test_split(
    wav_paths, texts, y, test_size=0.2, stratify=y, random_state=42
)

# MODELS (LOAD ONCE)
wav_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base")
wav_model = WavLMModel.from_pretrained("microsoft/wavlm-base").to(device)

for p in wav_model.parameters():
    p.requires_grad = False
wav_model.eval()

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
text_model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
text_model.eval()

# DATASET (ON-THE-FLY)
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, wavs, texts, labels):
        self.wavs = wavs
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.wavs)

    def __getitem__(self, idx):
        path = self.wavs[idx]
        text = self.texts[idx]
        label = self.labels[idx]

        # AUDIO (limit length)
        signal, _ = librosa.load(path, sr=16000)
        if len(signal) > 16000*6:
            signal = signal[:16000*6]

        inputs = wav_processor(signal, sampling_rate=16000,
                               return_tensors="pt", padding=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            a = wav_model(**inputs).last_hidden_state.mean(dim=1)

        # TEXT
        t_inputs = tokenizer(text, return_tensors="pt",
                             truncation=True, padding=True)
        t_inputs = {k:v.to(device) for k,v in t_inputs.items()}

        with torch.no_grad():
            t = text_model(**t_inputs).last_hidden_state[:,0,:]

        return a.squeeze(0).cpu(), t.squeeze(0).cpu(), label

# DATALOADER
train_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_train, Xt_train, y_train),
    batch_size=8, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_test, Xt_test, y_test),
    batch_size=8
)

# LOSS
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw = torch.tensor(cw, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

# MODEL
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.a_fc = nn.Linear(768,256)
        self.t_fc = nn.Linear(768,256)
        self.gate = nn.Linear(512,256)

        self.lstm = nn.LSTM(256,128,batch_first=True,bidirectional=True)

        self.cls = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,len(le.classes_))
        )

    def forward(self,a,t):
        a = self.a_fc(a)
        t = self.t_fc(t)

        g = torch.sigmoid(self.gate(torch.cat([a,t],dim=1)))
        fused = g*a + (1-g)*t

        fused = fused.unsqueeze(1)
        out,_ = self.lstm(fused)

        return self.cls(out[:,-1,:])

model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TRAIN
for epoch in range(6):
    model.train()
    total_loss = 0

    for a,t,yb in train_loader:
        a,t,yb = a.to(device), t.to(device), yb.to(device)

        out = model(a,t)
        loss = criterion(out,yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# EVAL
model.eval()
all_preds = []

with torch.no_grad():
    for a,t,_ in test_loader:
        a,t = a.to(device), t.to(device)
        preds = torch.argmax(model(a,t),1)
        all_preds.extend(preds.cpu().numpy())

print("\nAccuracy:", accuracy_score(y_test, all_preds))
print(classification_report(y_test, all_preds))

pred_labels = le.inverse_transform(all_preds)

# TREND ANALYSIS (UNCHANGED)
records = []

for idx, (path, pred) in enumerate(zip(Xa_test, pred_labels)):
    filename = os.path.basename(path).replace(".wav","")
    parts = filename.split("_")

    conv_id = "_".join(parts[:2])
    match = re.search(r'(\d+)$', filename)
    utt_id = int(match.group(1)) if match else idx

    records.append({
        "conversation_id": conv_id,
        "utterance_id": utt_id,
        "emotion_label": pred
    })

df = pd.DataFrame(records).sort_values(by=["conversation_id","utterance_id"])

VAD_MAP = {
    "hap": (0.8,0.7,0.6),
    "sad": (-0.7,0.3,0.4),
    "ang": (-0.6,0.8,0.7),
    "neu": (0.0,0.5,0.5)
}

print("\nSample Conversation IDs:")
print(df["conversation_id"].unique()[:10])

cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val,label="Valence")
    plt.plot(aro,"--",label="Arousal")
    plt.plot(dom,":",label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1]-val[0]
    var = np.var(val)

    print("\nCONCLUSION:")
    print("Positive Shift" if trend>0.2 else "Negative Shift" if trend<-0.2 else "Stable Emotion")
    print("High Fluctuation" if var>0.5 else "Stable Behavior")

else:
    print("Conversation not found")

In [ ]:
#  SOTA MULTIMODAL (WavLM + DistilBERT + Cross-Attention + BiLSTM)

!pip install transformers librosa -q

import os, re, glob, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import WavLMModel, AutoFeatureExtractor
from transformers import DistilBertTokenizer, DistilBertModel

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATA
DATA_PATH = "/content/IEMOCAP_full_release"
valid_emotions = ["ang","hap","sad","neu","exc"]

wav_paths, texts, labels = [], [], []

for session in range(1,6):
    session_path = os.path.join(DATA_PATH, f"Session{session}")
    wav_files = glob.glob(os.path.join(session_path, "**/*.wav"), recursive=True)

    for wav_file in wav_files:
        file_id = os.path.basename(wav_file).replace(".wav","")
        parts = file_id.split("_")

        ann_name = "_".join(parts[:2]) + ".txt" if "impro" in file_id else "_".join(parts[:3]) + ".txt"
        ann_path = os.path.join(session_path,"dialog","EmoEvaluation",ann_name)
        trans_path = os.path.join(session_path,"dialog","transcriptions",ann_name)

        if not os.path.exists(ann_path) or not os.path.exists(trans_path):
            continue

        emotion = None
        with open(ann_path) as f:
            for line in f:
                if file_id in line:
                    emotion = line.strip().split()[4]
                    break

        if emotion not in valid_emotions:
            continue
        if emotion == "exc":
            emotion = "hap"

        text = ""
        with open(trans_path) as f:
            for line in f:
                if file_id in line:
                    text = " ".join(line.strip().split()[2:])
                    break

        if text == "":
            continue

        wav_paths.append(wav_file)
        texts.append(text)
        labels.append(emotion)

print("Samples:", len(wav_paths))

# ENCODING
le = LabelEncoder()
y = le.fit_transform(labels)

Xa_train, Xa_test, Xt_train, Xt_test, y_train, y_test = train_test_split(
    wav_paths, texts, y, test_size=0.2, stratify=y, random_state=42
)

# MODELS
wav_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base")
wav_model = WavLMModel.from_pretrained("microsoft/wavlm-base").to(device)
for p in wav_model.parameters():
    p.requires_grad = False
wav_model.eval()

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
text_model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
text_model.eval()

# DATASET
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, wavs, texts, labels):
        self.wavs = wavs
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.wavs)

    def __getitem__(self, idx):
        path = self.wavs[idx]
        text = self.texts[idx]
        label = self.labels[idx]

        # AUDIO (trim)
        signal, _ = librosa.load(path, sr=16000)
        if len(signal) > 16000*6:
            signal = signal[:16000*6]

        inputs = wav_processor(signal, sampling_rate=16000,
                               return_tensors="pt", padding=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            a = wav_model(**inputs).last_hidden_state.mean(dim=1)

        # TEXT
        t_inputs = tokenizer(text, return_tensors="pt",
                             truncation=True, padding=True)
        t_inputs = {k:v.to(device) for k,v in t_inputs.items()}

        with torch.no_grad():
            t = text_model(**t_inputs).last_hidden_state[:,0,:]

        return a.squeeze(0).cpu(), t.squeeze(0).cpu(), label

# SAMPLER (IMBALANCE FIX)
class_counts = np.bincount(y_train)
weights = 1. / class_counts
sample_weights = weights[y_train]

sampler = torch.utils.data.WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_train, Xt_train, y_train),
    batch_size=8,
    sampler=sampler
)

test_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_test, Xt_test, y_test),
    batch_size=8
)

# LOSS
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw = torch.tensor(cw, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

# MODEL (SOTA)
class SOTA(nn.Module):
    def __init__(self):
        super().__init__()
        self.a_fc = nn.Linear(768,256)
        self.t_fc = nn.Linear(768,256)

        self.attn = nn.MultiheadAttention(256, 4, batch_first=True)

        self.lstm = nn.LSTM(256,128,batch_first=True,bidirectional=True)

        self.fc = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,len(le.classes_))
        )

    def forward(self,a,t):
        a = self.a_fc(a)
        t = self.t_fc(t)

        x = torch.stack([a,t], dim=1)  # (B,2,256)
        attn_out,_ = self.attn(x,x,x)

        lstm_out,_ = self.lstm(attn_out)
        out = lstm_out[:,-1,:]

        return self.fc(out)

model = SOTA().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scaler = torch.cuda.amp.GradScaler()

# TRAIN (FAST AMP)
for epoch in range(7):
    model.train()
    total_loss = 0

    for a,t,yb in train_loader:
        a,t,yb = a.to(device), t.to(device), yb.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(a,t)
            loss = criterion(out,yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# EVAL
model.eval()
all_preds = []

with torch.no_grad():
    for a,t,_ in test_loader:
        a,t = a.to(device), t.to(device)
        preds = torch.argmax(model(a,t),1)
        all_preds.extend(preds.cpu().numpy())

print("\nAccuracy:", accuracy_score(y_test, all_preds))
print(classification_report(y_test, all_preds))

pred_labels = le.inverse_transform(all_preds)

# EMOTION TREND ANALYSIS
records = []

for idx, (path, pred) in enumerate(zip(Xa_test, pred_labels)):
    filename = os.path.basename(path).replace(".wav","")
    parts = filename.split("_")

    conv_id = "_".join(parts[:2])
    match = re.search(r'(\d+)$', filename)
    utt_id = int(match.group(1)) if match else idx

    records.append({
        "conversation_id": conv_id,
        "utterance_id": utt_id,
        "emotion_label": pred
    })

df = pd.DataFrame(records).sort_values(by=["conversation_id","utterance_id"])

VAD_MAP = {
    "hap": (0.8,0.7,0.6),
    "sad": (-0.7,0.3,0.4),
    "ang": (-0.6,0.8,0.7),
    "neu": (0.0,0.5,0.5)
}

print("\nSample Conversation IDs:")
print(df["conversation_id"].unique()[:30])

cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val,label="Valence")
    plt.plot(aro,"--",label="Arousal")
    plt.plot(dom,":",label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1]-val[0]
    var = np.var(val)

    print("\nCONCLUSION:")
    print("Positive Shift" if trend>0.2 else "Negative Shift" if trend<-0.2 else "Stable Emotion")
    print("High Fluctuation" if var>0.5 else "Stable Behavior")

else:
    print("Conversation not found")

In [ ]:
cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val,label="Valence")
    plt.plot(aro,"--",label="Arousal")
    plt.plot(dom,":",label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1]-val[0]
    var = np.var(val)

    print("\nCONCLUSION:")
    print("Positive Shift" if trend>0.2 else "Negative Shift" if trend<-0.2 else "Stable Emotion")
    print("High Fluctuation" if var>0.5 else "Stable Behavior")

else:
    print("Conversation not found")

In [ ]:
cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val,label="Valence")
    plt.plot(aro,"--",label="Arousal")
    plt.plot(dom,":",label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1]-val[0]
    var = np.var(val)

    print("\nCONCLUSION:")
    print("Positive Shift" if trend>0.2 else "Negative Shift" if trend<-0.2 else "Stable Emotion")
    print("High Fluctuation" if var>0.5 else "Stable Behavior")

else:
    print("Conversation not found")

In [ ]:
# WavLM-Large + Transformer + Attention Pooling

!pip install transformers librosa -q

import os, re, glob, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import WavLMModel, AutoFeatureExtractor

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# LOAD DATA
DATA_PATH = "/content/IEMOCAP_full_release"
valid_emotions = ["ang","hap","sad","neu","exc"]

wav_paths, labels = [], []

for session in range(1,6):
    session_path = os.path.join(DATA_PATH, f"Session{session}")
    wav_files = glob.glob(os.path.join(session_path, "**/*.wav"), recursive=True)

    for wav_file in wav_files:
        file_id = os.path.basename(wav_file).replace(".wav","")
        parts = file_id.split("_")

        ann_name = "_".join(parts[:2]) + ".txt" if "impro" in file_id else "_".join(parts[:3]) + ".txt"
        ann_path = os.path.join(session_path,"dialog","EmoEvaluation",ann_name)

        if not os.path.exists(ann_path):
            continue

        emotion = None
        with open(ann_path) as f:
            for line in f:
                if file_id in line:
                    emotion = line.strip().split()[4]
                    break

        if emotion not in valid_emotions:
            continue
        if emotion == "exc":
            emotion = "hap"

        wav_paths.append(wav_file)
        labels.append(emotion)

print("Samples:", len(wav_paths))

# ENCODING
le = LabelEncoder()
y = le.fit_transform(labels)

Xa_train, Xa_test, y_train, y_test = train_test_split(
    wav_paths, y, test_size=0.2, stratify=y, random_state=42
)

# LOAD WAVLM-LARGE (FROZEN)
wav_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
wav_model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(device)

for p in wav_model.parameters():
    p.requires_grad = False

wav_model.eval()

# DATASET (MEMORY SAFE)
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, wavs, labels):
        self.wavs = wavs
        self.labels = labels

    def __len__(self):
        return len(self.wavs)

    def __getitem__(self, idx):
        path = self.wavs[idx]
        label = self.labels[idx]

        # LOAD AUDIO (STRICT TRIM)
        signal, _ = librosa.load(path, sr=16000)

        max_len = 16000 * 5   # 5 sec limit (VERY IMPORTANT)
        if len(signal) > max_len:
            signal = signal[:max_len]

        inputs = wav_processor(
            signal,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        inputs = {k:v.to(device) for k,v in inputs.items()}

        with torch.no_grad():
            out = wav_model(**inputs).last_hidden_state  # (1, T, 1024)

        return out.squeeze(0).cpu(), label


# COLLATE (PADDING FOR VARIABLE LENGTH)
def collate_fn(batch):
    feats, labels = zip(*batch)

    lengths = [f.shape[0] for f in feats]
    max_len = min(max(lengths), 400)  # HARD LIMIT (prevents crash)

    padded = []
    for f in feats:
        if f.shape[0] > max_len:
            f = f[:max_len]
        else:
            pad = torch.zeros(max_len - f.shape[0], f.shape[1])
            f = torch.cat([f, pad], dim=0)
        padded.append(f)

    return torch.stack(padded), torch.tensor(labels)


# IMBALANCE HANDLING
class_counts = np.bincount(y_train)
weights = 1. / class_counts
sample_weights = weights[y_train]

sampler = torch.utils.data.WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_train, y_train),
    batch_size=4,   # LOW batch (important)
    sampler=sampler,
    collate_fn=collate_fn
)

test_loader = torch.utils.data.DataLoader(
    EmotionDataset(Xa_test, y_test),
    batch_size=4,
    collate_fn=collate_fn
)

# CLASS WEIGHTS LOSS
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw = torch.tensor(cw, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)

# MODEL (Transformer + Attention Pooling)
class AudioSOTA(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(1024, 256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Attention Pooling
        self.attn_pool = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, len(le.classes_))
        )

    def forward(self, x):
        x = self.input_proj(x)

        x = self.transformer(x)

        # MEAN POOL
        mean_pool = torch.mean(x, dim=1)

        # ATTENTION POOL
        attn_weights = torch.softmax(self.attn_pool(x), dim=1)
        attn_pool = torch.sum(attn_weights * x, dim=1)

        combined = torch.cat([mean_pool, attn_pool], dim=1)

        return self.fc(combined)


model = AudioSOTA().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scaler = torch.cuda.amp.GradScaler()

# TRAIN
for epoch in range(8):
    model.train()
    total_loss = 0

    for x, yb in train_loader:
        x, yb = x.to(device), yb.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

    gc.collect()
    torch.cuda.empty_cache()

# EVAL
model.eval()
all_preds = []

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        preds = torch.argmax(model(x),1)
        all_preds.extend(preds.cpu().numpy())

print("\nAccuracy:", accuracy_score(y_test, all_preds))
print(classification_report(y_test, all_preds))

pred_labels = le.inverse_transform(all_preds)

# EMOTION TREND

records = []

for idx, (path, pred) in enumerate(zip(Xa_test, pred_labels)):
    filename = os.path.basename(path).replace(".wav","")
    parts = filename.split("_")

    conv_id = "_".join(parts[:2])
    match = re.search(r'(\d+)$', filename)
    utt_id = int(match.group(1)) if match else idx

    records.append({
        "conversation_id": conv_id,
        "utterance_id": utt_id,
        "emotion_label": pred
    })

df = pd.DataFrame(records).sort_values(by=["conversation_id","utterance_id"])

VAD_MAP = {
    "hap": (0.8,0.7,0.6),
    "sad": (-0.7,0.3,0.4),
    "ang": (-0.6,0.8,0.7),
    "neu": (0.0,0.5,0.5)
}

print("\nSample Conversation IDs:")
print(df["conversation_id"].unique()[:30])

cid = input("\nEnter conversation ID: ")

subset = df[df["conversation_id"] == cid]

if len(subset) > 0:
    emotions = subset["emotion_label"].tolist()
    vad = np.array([VAD_MAP[e] for e in emotions])

    val = gaussian_filter1d(vad[:,0],1)
    aro = gaussian_filter1d(vad[:,1],1)
    dom = gaussian_filter1d(vad[:,2],1)

    plt.plot(val,label="Valence")
    plt.plot(aro,"--",label="Arousal")
    plt.plot(dom,":",label="Dominance")

    plt.legend()
    plt.title(cid)
    plt.show()

    trend = val[-1]-val[0]
    var = np.var(val)

    print("\nCONCLUSION:")
    print("Positive Shift" if trend>0.2 else "Negative Shift" if trend<-0.2 else "Stable Emotion")
    print("High Fluctuation" if var>0.5 else "Stable Behavior")

else:
    print("Conversation not found")